In [ ]:
import os

import chemiscope
import ipi
import numpy as np
from ase.visualize import view
from matplotlib import pyplot as plt
import nqetools as nqe
# This follows:
# https://github.com/i-pi/piqm2023-tutorial/blob/main/05-RPI/tutorial-4.ipynb

In [ ]:
#UNITS:
invcm2au = 4.5563353e-06  #in i-PI unit_to_internal("frequency", "inversecm", 1.0)
kelvin2au = 3.1668152e-06


def get_freq_from_eigvals(eigvals):
    """Transforms eigenvalues in atomic units to frequencies in cm^-1"""
    freq_invcm = np.zeros(eigvals.shape)
    for i, eig in enumerate(eigvals):
        freq_invcm[i] = np.sign(eig) * np.absolute(eig) ** 0.5 / invcm2au
    return freq_invcm


# Paths
directory_opti = 'opti'
directory_phonon_react = 'phonon_react'
directory_ts = 'ts'
directory_phonon_ts = 'phonon_ts'
directory_instanton = 'instanton'

# Driver
driver_code = 'cbe'

# Values
temperature = 300.0
n_beads = 10
tol_energy = 5.0e-6
tol_force = 5.0e-6
tol_position = 1.0e-6
total_steps = 1000
optimizer = "cg"


In [ ]:
atoms = nqe.read_ipi_xyz("react.xyz")[-1]
n_atoms = len(atoms)

In [ ]:
# Run minimization
output = nqe.run_optimise(directory_opti, atoms, driver=driver_code)
atoms_opti, output_data_opti, output_desc_opti = output

In [ ]:
# Plot the energy of the minimisation
nqe.plot_step_energy(output_data_opti)

In [ ]:
nqe.run_phonons(directory_phonon_react, atoms_opti, driver=driver_code)

In [ ]:
eigvals = np.genfromtxt(os.path.join(directory_phonon_react, 'phonon.phonons.eigval'))
fr = get_freq_from_eigvals(eigvals)

plt.plot(fr, 'o')
plt.xlabel('Vibrational Mode  Index')
plt.ylabel('Frequency (cm$^{-1}$)')
plt.show()

In [ ]:
output = nqe.run_ts(directory_ts, atoms_opti, driver=driver_code)
atoms_ts, output_data_ts, output_desc_ts = output

In [ ]:
# Plot the energy of the minimisation
nqe.plot_step_energy(output_data_ts)

In [ ]:
nqe.run_phonons(directory_phonon_ts, atoms_ts, driver=driver_code)

In [ ]:
eigvals = np.genfromtxt(os.path.join(directory_phonon_ts, 'phonon.phonons.eigval'))
fr = get_freq_from_eigvals(eigvals)

plt.plot(fr, 'o')
plt.xlabel('Vibrational Mode  Index')
plt.ylabel('Frequency (cm$^{-1}$)')
plt.show()

In [ ]:
# Run the instanton
nqe.run_instanton(directory_instanton, atoms_ts, directory_phonon_ts, driver=driver_code, n_beads=n_beads,
                  temperature=temperature)

In [ ]:

# Run the instanton analysis for the reactant
nqe.instanton_postproc(os.path.join(directory_phonon_react, "RESTART"),
                       case="reactant",
                       temperature=temperature,
                       n_beads_r=n_beads,
                       filter_list=[n_atoms - 1])

In [ ]:
nqe.calc_kappa(os.path.join(directory_phonon_ts, "RESTART"),
               os.path.join(directory_instanton, "RESTART"),
               temperature,
               n_beads)